In [ ]:
# [NB03-C01] SETUP
# What: install the pinned pm4py version, import the libraries and create
#       the figures folder.
# Why: same environment as NB01/NB02 so the three notebooks agree; the
#      version is pinned for Colab.
# Expected: no errors.

%pip install pm4py==2.7.23.1 -q

import os                                       # filesystem helpers
import pandas as pd                             # dataframes and CSV handling
import numpy as np                              # numeric helpers
import matplotlib.pyplot as plt                 # plots
import seaborn as sns                           # heatmap of the quality pillars
import pm4py                                    # process mining library of the course
from pm4py.util import constants                # to silence the replay progress bars
from sklearn.model_selection import train_test_split   # case-level hold-out
from scipy.stats import kruskal                 # non-parametric test (chosen in NB01)

# Silence the token-replay progress bars: they would flood the notebook with
# thousands of lines, hiding the actual results
constants.SHOW_PROGRESS_BAR = False

# Fixed seed: the split and every result of this notebook are reproducible
RANDOM_STATE = 42

# Create the folder where the report figures will be saved (safe if it exists)
os.makedirs('figures', exist_ok=True)

In [ ]:
# [NB03-C02] LOAD THE PREPARED FILES
# What: load the abstract control-flow view, the case table and the two
#       segment selection lists produced by NB02.
# Why: discovery and conformance work on the control flow, so the abstract
#      view is the right input (the raw view would create artificial loops
#      from same-instant recordings, see NB01-C16). NO row is dropped here.
# Expected: 16,826 abstract rows, 1,820 stays, 447 head stays, 723 tail stays.

# Paths of the prepared files (on Colab: upload them and update the paths)
abstract_path = 'abstract_event_log.csv'
cases_path = 'case_table.csv'
segment_a_path = 'segment_A_cases.csv'
segment_c_path = 'segment_C_cases.csv'

# Defensive check: stop with a clear message if a file is missing
for p in [abstract_path, cases_path, segment_a_path, segment_c_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(p + ' not found - run NB01 and NB02 first.')

# Load the abstract control-flow view (case ids as strings: pm4py convention)
abstract_log = pd.read_csv(abstract_path, parse_dates=['time:timestamp'])
abstract_log['case:concept:name'] = abstract_log['case:concept:name'].astype(str)

# Load the case table (one row per stay)
case_table = pd.read_csv(cases_path, index_col=0, parse_dates=['start_time', 'end_time'])
case_table.index = case_table.index.astype(str)

# Load the segment selection lists (pointers into the log, not copies)
segment_a = pd.read_csv(segment_a_path, index_col=0)
segment_a.index = segment_a.index.astype(str)
segment_c = pd.read_csv(segment_c_path, index_col=0)
segment_c.index = segment_c.index.astype(str)

print('Abstract view rows: {} (expected 16,826)'.format(len(abstract_log)))
print('Stays in the case table: {} (expected 1,820)'.format(len(case_table)))
print('Head stays (segment A): {} | long-tail stays (segment C): {}'.format(len(segment_a), len(segment_c)))

In [ ]:
# [NB03-C03] SANITY CHECK: START AND END ACTIVITIES
# What: re-check the log boundaries after the reload.
# Why: course practice - the boundaries are verified after every load,
#      never assumed.
# Expected: 1,820 "Enter the ED" starts and 1,820 "Discharge from the ED" ends.

# Start and end activities of the abstract view
print('Start activities: {}'.format(pm4py.get_start_activities(abstract_log)))
print('End activities: {}'.format(pm4py.get_end_activities(abstract_log)))

# Interpretation: the log is intact, all 1,820 stays are complete.

In [ ]:
# [NB03-C04] THE EXPECTED PROCESS FROM THE SCENARIO (A-PRIORI MODEL)
# What: formalise the patient journey DESCRIBED in the case-study text as a
#       process model, before looking at anything discovered from data, and
#       render it in the four course notations: process tree, Petri net,
#       BPMN and flow diagram.
# Why: the scenario provides no formal protocol, but its Description implies
#      an expected flow: "arrival and initial assessment, clinical
#      evaluation, execution of necessary clinical procedures, and finally
#      patient discharge or referral". Making this explicit BEFORE discovery
#      gives the comparison the assignment asks for: expected (from the
#      text) vs observed (from the data).
# Expected: four figures saved for the report.

# The expected process as a process tree: a sequence Enter -> Triage, then
# zero or more clinical activities (any of the three, in any order: the
# scenario says procedures are executed "as necessary"), then Discharge.
expected_tree = pm4py.parse_process_tree(
    "->( 'Enter the ED', 'Triage in the ED', "
    "*( tau, X( 'Medicine reconciliation', 'Medicine dispensations', 'Vital sign check' ) ), "
    "'Discharge from the ED' )")
print('Expected process tree: {}'.format(expected_tree))

# 1. Process tree view (the native notation of the definition above)
pm4py.save_vis_process_tree(expected_tree, 'figures/NB03_expected_process_tree.png')
pm4py.view_process_tree(expected_tree)

# 2. Petri net view (execution semantics, used by the conformance below)
expected_net, expected_im, expected_fm = pm4py.convert_to_petri_net(expected_tree)
pm4py.save_vis_petri_net(expected_net, expected_im, expected_fm, 'figures/NB03_expected_petri_net.png')

# 3. BPMN view (the business-facing notation)
expected_bpmn = pm4py.convert_to_bpmn(expected_tree)
pm4py.save_vis_bpmn(expected_bpmn, 'figures/NB03_expected_bpmn.png')

# 4. Flow diagram (a plain flowchart for non-technical readers)
import graphviz
flow = graphviz.Digraph()
flow.attr(rankdir='TB')
flow.node('start', 'Patient arrival', shape='oval')                       # start event
flow.node('enter', 'Enter the ED', shape='box')                           # registration
flow.node('triage', 'Triage in the ED', shape='box')                      # initial assessment
flow.node('clinical', 'Clinical work (reconciliation / dispensations / vital sign checks)', shape='box')
flow.node('more', 'More clinical work needed?', shape='diamond')   # repeat decision
flow.node('discharge', 'Discharge from the ED (home, admission or referral)', shape='box')
flow.node('end', 'Stay closed', shape='oval')                             # end event
flow.edge('start', 'enter')
flow.edge('enter', 'triage')
flow.edge('triage', 'clinical')
flow.edge('clinical', 'more')
flow.edge('more', 'clinical', label='yes')
flow.edge('more', 'discharge', label='no')
flow.edge('discharge', 'end')
flow.render('figures/NB03_expected_flow_diagram', format='png', cleanup=True)
print('Saved: expected process tree, Petri net, BPMN and flow diagram in figures/')

# Interpretation: this model encodes ONLY what the scenario text says. It is
# deliberately permissive (any mix of clinical work is allowed): how much
# more permissive than reality, and where reality escapes even this
# description, is measured against the data in [NB03-C13].

In [ ]:
# [NB03-C05] TRAIN / TEST SPLIT AT CASE LEVEL
# What: split the stays (not the events) into a 70% train set used to
#       discover models and a 30% test set kept aside for the final
#       evaluation.
# Why: a model evaluated on the same data that produced it always looks
#      better than it is. Splitting at CASE level (never at event level)
#      keeps every trace whole. The split is a hold-out for honest
#      evaluation, NOT a filter: no stay is removed from the project, both
#      halves stay in the log and are reunited for the conformance
#      diagnostics.
# Expected: ~1,274 train stays and ~546 test stays, same disposition mix.

# Coarse disposition used only to stratify the split, so that rare outcomes
# (e.g. EXPIRED, 3 stays) do not end up entirely on one side
def coarse_disposition(value):
    if value in ['HOME', 'ADMITTED']:
        return value
    return 'OTHER'
strata = case_table['disposition'].apply(coarse_disposition)

# Split the case ids (strings! pm4py case ids are strings and mixing types
# would silently produce empty filters)
all_case_ids = case_table.index.to_numpy()
train_ids, test_ids = train_test_split(all_case_ids, test_size=0.30,
                                       random_state=RANDOM_STATE, stratify=strata)

# Build the two logs by selecting the events of each group of stays
train_log = abstract_log[abstract_log['case:concept:name'].isin(train_ids)]
test_log = abstract_log[abstract_log['case:concept:name'].isin(test_ids)]

# Verify the split: sizes, and that no stay is lost or shared
print('Train stays: {} | Test stays: {} | Total: {}'.format(
    len(train_ids), len(test_ids), len(train_ids) + len(test_ids)))
print('Train events: {} | Test events: {} | Total: {} (expected 16,826)'.format(
    len(train_log), len(test_log), len(train_log) + len(test_log)))
print('Stays in both sets (must be 0): {}'.format(len(set(train_ids) & set(test_ids))))

# Verify the disposition mix is preserved on both sides
print('\nDisposition mix (share of stays):')
mix = pd.DataFrame({'train': case_table.loc[train_ids, 'disposition'].value_counts(normalize=True),
                    'test': case_table.loc[test_ids, 'disposition'].value_counts(normalize=True)})
print(mix.round(3))

# Interpretation: the two halves have the same outcome composition, so the
# test set is a fair judge of a model discovered on the train set.

In [ ]:
# [NB03-C06] HELPER: DISCOVER AND EVALUATE A MODEL ON THE FOUR QUALITY PILLARS
# What: two small helpers used by every experiment below: one discovers an
#       Inductive Miner model, the other measures a model on a given log with
#       the four quality dimensions of the course (Module 3.1): FITNESS,
#       PRECISION, GENERALIZATION and SIMPLICITY, plus the F1 that combines
#       the first two.
# Why: the same measurement code must serve every candidate, otherwise the
#      comparison is unfair (course lesson: comparing models evaluated with
#      different procedures is meaningless).
# Expected: no output - these are definitions.

# F1 with beta: beta > 1 prioritises recall (fitness), which in process
# mining means "the model must be able to replay the real behaviour".
# The course uses beta = 1.5 (Case6, Case8).
BETA = 1.5

# Discover a Petri net with the Inductive Miner on a given log
def discover_inductive(log, noise_threshold):
    net, im, fm = pm4py.discover_petri_net_inductive(log, noise_threshold=noise_threshold)
    return net, im, fm

# Measure fitness, precision and F1 of a model on a given log
def evaluate_model(log, net, im, fm, beta=BETA):
    # Token-based replay fitness: how much of the real behaviour the model replays
    fitness_result = pm4py.fitness_token_based_replay(log, net, im, fm)
    # Defensive extraction: the key name changed across pm4py versions
    fitness = fitness_result.get('average_trace_fitness', fitness_result.get('log_fitness', np.nan))
    # Precision: how much behaviour the model allows that never happens
    precision = pm4py.precision_token_based_replay(log, net, im, fm)
    # F1 with beta; the small constant avoids a division by zero
    f1 = (1 + beta ** 2) * precision * fitness / (beta ** 2 * precision + fitness + 1e-12)
    # Generalization: does the model rely on behaviour seen only once, or does
    # it generalise to unseen-but-similar behaviour? (third quality pillar)
    generalization = pm4py.generalization_tbr(log, net, im, fm)
    # Simplicity: how readable is the net structure? (fourth quality pillar,
    # Occam's razor: among equally good models prefer the simpler one)
    simplicity = pm4py.simplicity_petri_net(net, im, fm)
    # Number of real activities in the net (a flower model would show all of them
    # with almost no structure, so we track it as a readability signal)
    activities = [t.label for t in net.transitions if t.label is not None]
    return {'fitness': fitness, 'precision': precision, 'f1': f1,
            'generalization': generalization, 'simplicity': simplicity,
            'activities_in_net': len(activities),
            'transitions': len(net.transitions), 'places': len(net.places), 'arcs': len(net.arcs)}

print('Helpers defined. F1 uses beta = {} (fitness prioritised, as in Case6/Case8).'.format(BETA))

In [ ]:
# [NB03-C07] STRATEGY 1 - INDUCTIVE MINER ON THE WHOLE TRAIN SET (NOISE GRID)
# What: discover a model on ALL the train stays, varying the noise_threshold
#       of the Inductive Miner, and measure each candidate on the train set.
# Why: this is the option that removes NO case: the noise_threshold does not
#      delete stays, it lets the miner down-weight infrequent behaviour when
#      building the process tree. Given that the variant distribution has no
#      clean Pareto head (NB02-C16), this is the natural first candidate.
# Expected: fitness decreasing and precision increasing as the threshold grows.

# Grid of noise thresholds to try (0 = keep all behaviour in the model)
noise_grid = [0.0, 0.1, 0.2, 0.3, 0.4]

# Try each threshold and measure it on the train set
noise_rows = []
for noise in noise_grid:
    net, im, fm = discover_inductive(train_log, noise)
    scores = evaluate_model(train_log, net, im, fm)
    scores['noise_threshold'] = noise
    scores['cases_used_for_mining'] = len(train_ids)   # always ALL train stays
    noise_rows.append(scores)
    print('noise={:.1f} -> fitness {:.3f} | precision {:.3f} | F1 {:.3f} | activities {} | places {} | arcs {}'.format(
        noise, scores['fitness'], scores['precision'], scores['f1'],
        scores['activities_in_net'], scores['places'], scores['arcs']))

noise_table = pd.DataFrame(noise_rows)

# Interpretation: read the trade-off column by column. A model with fitness
# near 1 and very low precision is close to a "flower model" (it allows
# almost everything and explains nothing); a model with high precision and
# low fitness is too rigid to describe the real ED.

In [ ]:
# [NB03-C08] STRATEGY 2 - FIND_BEST_K_BY_F1 (COURSE METHOD)
# What: the course helper: for each k, keep the k most frequent variants of
#       the train set, discover a model on them, and measure it on the FULL
#       train set (not on the filtered subset).
# Why: this is how Case3, Case4, Case6 and Case8 choose how many variants a
#      readable model should describe: the choice is task-driven (best F1),
#      not distribution-driven. Note: the filtering here is TEMPORARY and
#      internal to the experiment - it selects what the miner SEES, while
#      the evaluation always uses every train stay. No stay is removed from
#      the project.
# Expected: F1 rising with k up to a plateau; the k chosen by F1 is the
#      candidate to compare with the head size of NB02 (12 variants).

# Candidate numbers of variants to give the miner
k_values = [1, 2, 3, 5, 8, 10, 12, 15, 20, 30, 50]

# Baseline: the model discovered on the whole train set (k = all variants)
base_net, base_im, base_fm = discover_inductive(train_log, 0.0)
baseline = evaluate_model(train_log, base_net, base_im, base_fm)
print('Baseline (all {} train variants, noise=0): fitness {:.3f} | precision {:.3f} | F1 {:.3f}\n'.format(
    train_log.groupby('case:concept:name')['concept:name'].agg(tuple).nunique(),
    baseline['fitness'], baseline['precision'], baseline['f1']))

# Try each k
k_rows = []
for k in k_values:
    # Keep the k most frequent variants: what the MINER sees
    rep_log = pm4py.filter_variants_top_k(train_log, k)
    # How many train stays are covered by those k variants
    cases_covered = rep_log['case:concept:name'].nunique()
    # Discover on the representative subset
    net, im, fm = discover_inductive(rep_log, 0.0)
    # Measure on ALL the train stays (the honest denominator)
    scores = evaluate_model(train_log, net, im, fm)
    scores['k'] = k
    scores['train_cases_covered'] = cases_covered
    scores['train_coverage_pct'] = round(cases_covered / len(train_ids) * 100, 1)
    k_rows.append(scores)
    print('k={:2d} -> covers {:4d}/{} train stays ({:4.1f}%) | fitness {:.3f} | precision {:.3f} | F1 {:.3f} | activities {}'.format(
        k, cases_covered, len(train_ids), scores['train_coverage_pct'],
        scores['fitness'], scores['precision'], scores['f1'], scores['activities_in_net']))

k_table = pd.DataFrame(k_rows)

# The k with the best F1 (the course criterion)
best_row = k_table.loc[k_table['f1'].idxmax()]
print('\nBest k by F1: k = {:.0f} (F1 = {:.3f}, fitness = {:.3f}, precision = {:.3f})'.format(
    best_row['k'], best_row['f1'], best_row['fitness'], best_row['precision']))

In [ ]:
# [NB03-C09] STRATEGY 3 - DISCOVERY ON THE HEAD SEGMENT (NB02)
# What: discover a model on the head variants defined in NB02 (>= 20 cases)
#       restricted to the train stays, and measure it on the full train set.
# Why: the head is the segment the course reads as "the standard paths".
#      This candidate asks: does a model built only on the recognisable
#      routines describe the ED well enough? It is the third option to put
#      on the table before any decision.
# Expected: high precision (few paths) and lower fitness than the baseline.

# Head stays that belong to the train set (intersection of two selections)
head_train_ids = [case_id for case_id in segment_a.index if case_id in set(train_ids)]
head_train_log = abstract_log[abstract_log['case:concept:name'].isin(head_train_ids)]
print('Head stays in the train set: {}'.format(len(head_train_ids)))

# Discover on the head stays only (what the miner sees)
head_net, head_im, head_fm = discover_inductive(head_train_log, 0.0)

# Measure on ALL the train stays (again, the honest denominator)
head_scores = evaluate_model(train_log, head_net, head_im, head_fm)
print('Head-based model: fitness {:.3f} | precision {:.3f} | F1 {:.3f} | activities {} | places {} | arcs {}'.format(
    head_scores['fitness'], head_scores['precision'], head_scores['f1'],
    head_scores['activities_in_net'], head_scores['places'], head_scores['arcs']))

# Interpretation: compare this line with the k-table above; the head model is
# essentially the "k = 12 variants" candidate, seen from the segmentation
# side instead of the frequency-ranking side.

In [ ]:
# [NB03-C10] CROSS-CHECK 1 OF THE NB02 HEAD THRESHOLD: DOES THE TASK AGREE?
# What: compare the k chosen by F1 (task-driven, this notebook) with the
#       size of the head segment declared in NB02 (12 variants, n >= 20).
# Why: NB02 declared the threshold provisional and promised this check: two
#      independent criteria - one statistical/business (n >= 20 to read a
#      p90 inside the variant), one task-driven (best F1 of the model) -
#      should land in the same region if the head is a real structure.
# Expected: convergence would confirm the threshold; a large gap must be
#      reported honestly.

# The two numbers to compare
k_by_f1 = int(best_row['k'])
head_size = segment_a['variant_id'].nunique()
print('k chosen by F1 (task-driven): {}'.format(k_by_f1))
print('Head size declared in NB02 (n >= 20 cases): {} variants'.format(head_size))

# F1 of the model built on exactly the head-size number of variants, to see
# how much the choice would cost in F1 terms
closest = k_table.iloc[(k_table['k'] - head_size).abs().argmin()]
print('\nF1 at the best k       : {:.3f} (k = {:.0f})'.format(best_row['f1'], best_row['k']))
print('F1 at the NB02 head size: {:.3f} (k = {:.0f})'.format(closest['f1'], closest['k']))
print('F1 difference: {:.3f}'.format(abs(best_row['f1'] - closest['f1'])))

In [ ]:
# [NB03-C11] CROSS-CHECK 2 OF THE NB02 HEAD THRESHOLD: DO THE SEGMENTS DIFFER?
# What: test whether the three variant segments of NB02 (head, torso, long
#       tail) really differ in stay duration, with a Kruskal-Wallis test.
# Why: the second promised check. A segmentation that only sorts variants by
#      frequency is bookkeeping; if the segments also behave differently on
#      a KPI, the split captures something real about the process.
# Expected: a significant difference would support the segmentation.

# Attach the segment of each stay to its lead time. Stays of the torso are
# the ones in neither selection list (no data is filtered: every stay gets
# a label)
def stay_segment(case_id):
    if case_id in set(segment_a.index):
        return 'A - head'
    if case_id in set(segment_c.index):
        return 'C - long tail'
    return 'B - torso'
case_table['variant_segment'] = case_table.index.map(stay_segment)

# Median and p90 lead time per segment, with group sizes (denominators)
def p90(values):
    return values.quantile(0.90)
by_segment = case_table.groupby('variant_segment')['lead_time_min'].agg(['size', 'median', p90])
by_segment.columns = ['stays', 'median_min', 'p90_min']
print('Stay duration by variant segment (minutes):')
print(by_segment)

# Kruskal-Wallis across the three segments (non-parametric, as decided in NB01)
segment_groups = []
for segment_name, group in case_table.groupby('variant_segment'):
    segment_groups.append(group['lead_time_min'].dropna().values)
stat, p_value = kruskal(*segment_groups)
print('\nKruskal-Wallis H = {:.2f}, p-value = {:.6f}'.format(stat, p_value))
if p_value < 0.05:
    print('The three segments differ significantly in stay duration: the segmentation separates behaviour, not just frequency.')
else:
    print('No significant duration difference across segments: the segmentation is only a frequency ranking.')

In [ ]:
# [NB03-C12] DECISION POINT: THE CANDIDATES ON ONE TABLE
# What: put the three strategies side by side, with the counts before/after
#       that the project rule requires, and stop.
# Why: project rule - no filter or representative selection is ADOPTED
#      without the approval of the project owner. Everything above is
#      measurement on candidates; nothing has been removed from the data.
#      This table is the proposal.
# Expected: a table to read together, then a decision.

# Build the comparison table of the three strategies
candidates = []
# Strategy 1: the best noise threshold by F1, no case filtered
best_noise = noise_table.loc[noise_table['f1'].idxmax()]
candidates.append({'strategy': '1 - Inductive on ALL train stays (noise={:.1f})'.format(best_noise['noise_threshold']),
                   'stays_seen_by_miner': len(train_ids),
                   'pct_of_train_seen': 100.0,
                   'fitness': round(best_noise['fitness'], 3),
                   'precision': round(best_noise['precision'], 3),
                   'f1': round(best_noise['f1'], 3),
                   'places': int(best_noise['places']), 'arcs': int(best_noise['arcs'])})
# Strategy 2: the best k by F1
candidates.append({'strategy': '2 - Inductive on top-{:.0f} variants (find_best_k_by_f1)'.format(best_row['k']),
                   'stays_seen_by_miner': int(best_row['train_cases_covered']),
                   'pct_of_train_seen': best_row['train_coverage_pct'],
                   'fitness': round(best_row['fitness'], 3),
                   'precision': round(best_row['precision'], 3),
                   'f1': round(best_row['f1'], 3),
                   'places': int(best_row['places']), 'arcs': int(best_row['arcs'])})
# Strategy 3: the head segment of NB02
candidates.append({'strategy': '3 - Inductive on the NB02 head segment (n >= 20)',
                   'stays_seen_by_miner': len(head_train_ids),
                   'pct_of_train_seen': round(len(head_train_ids) / len(train_ids) * 100, 1),
                   'fitness': round(head_scores['fitness'], 3),
                   'precision': round(head_scores['precision'], 3),
                   'f1': round(head_scores['f1'], 3),
                   'places': int(head_scores['places']), 'arcs': int(head_scores['arcs'])})
decision_table = pd.DataFrame(candidates)

# Save the tables for the report
noise_table.to_csv('model_selection_noise.csv', index=False)
k_table.to_csv('model_selection_topk.csv', index=False)
decision_table.to_csv('model_selection_candidates.csv', index=False)
print('Saved model_selection_noise.csv, model_selection_topk.csv, model_selection_candidates.csv\n')

# NO-DATA-LOSS CHECK: every stay is still here, in the train or in the test
print('Stays in train + test: {} (started with {})'.format(len(train_ids) + len(test_ids), len(case_table)))
print('Events in train + test: {} (abstract view has {})'.format(len(train_log) + len(test_log), len(abstract_log)))
print('\nNo variant, stay or event has been removed from the project. The')
print('filtering inside strategies 2 and 3 only decides what the MINER SEES;')
print('every evaluation above uses all the train stays.')
print('\nAWAITING APPROVAL: the final model of NB03 will be discovered only')
print('after the project owner picks one of these strategies.')

decision_table

In [ ]:
# [NB03-C13] HONEST COMPARISON ON THE TEST SET (FOUR QUALITY PILLARS)
# What: discover the three candidate models on the train set and evaluate
#       them - plus the expected model of NB03-C04 - on the held-out test
#       set, on all four quality dimensions of Module 3.1.
# Why: the decision was to let the test metrics choose, not the train ones.
#      The train scores of NB03-C12 tell how each model fits the data it
#      saw; the test scores tell how each model GENERALISES to stays it
#      never saw - the only fair tie-breaker.
# Expected: the ranking may differ from the train ranking; that is exactly
#      why we hold out a test set.

# Re-discover the three candidate models on the TRAIN set
# Strategy 1: Inductive on all train stays, best noise from NB03-C07 (0.1)
model1 = discover_inductive(train_log, 0.1)
# Strategy 2: Inductive on the top-20 train variants (best k by F1)
rep20 = pm4py.filter_variants_top_k(train_log, 20)
model2 = discover_inductive(rep20, 0.0)
# Strategy 3: Inductive on the head-segment train stays (n >= 20)
head_train_ids = [case_id for case_id in segment_a.index if case_id in set(train_ids)]
head_train_log = abstract_log[abstract_log['case:concept:name'].isin(head_train_ids)]
model3 = discover_inductive(head_train_log, 0.0)

# Evaluate each model on the TEST set (the honest hold-out)
test_rows = []
for name, model in [('0 - expected model (scenario text)', (expected_net, expected_im, expected_fm)),
                    ('1 - all train stays (noise=0.1)', model1),
                    ('2 - top-20 variants (F1)', model2),
                    ('3 - head segment (n>=20)', model3)]:
    net, im, fm = model
    scores = evaluate_model(test_log, net, im, fm)
    scores['strategy'] = name
    test_rows.append(scores)
    print('{:34s} -> fitness {:.3f} | precision {:.3f} | generalization {:.3f} | simplicity {:.3f} | F1 {:.3f}'.format(
        name, scores['fitness'], scores['precision'], scores['generalization'], scores['simplicity'], scores['f1']))

test_table = pd.DataFrame(test_rows)[['strategy', 'fitness', 'precision', 'generalization', 'simplicity', 'f1', 'places', 'arcs']]
test_table.to_csv('model_selection_test.csv', index=False)
print('\nSaved model_selection_test.csv')

# Pick the winner by TEST F1 among the DISCOVERED strategies (the expected
# model is a benchmark from the scenario text, not a data-driven candidate:
# the empirical reference must describe typical observed behaviour)
discovered = test_table[~test_table['strategy'].str.startswith('0')]
winner_idx = discovered['f1'].idxmax()
winner_name = discovered.loc[winner_idx, 'strategy']
print('Winner by test F1: {}'.format(winner_name))

# Underfitting / overfitting reading of the trade-off (course framing):
# high fitness + low precision = the model is too general (underfitting);
# low fitness + high precision = the model is too rigid (overfitting);
# a balanced F1 marks the best trade-off.
print('\nFitness-precision trade-off (course framing):')
for row in test_rows:
    if row['fitness'] >= 0.95 and row['precision'] < 0.80:
        label = 'UNDERFITTING (too general: replays everything, constrains little)'
    elif row['fitness'] < 0.90 and row['precision'] >= 0.90:
        label = 'OVERFITTING (too rigid: precise but cannot replay the log)'
    else:
        label = 'balanced trade-off'
    print('  {:34s} -> {}'.format(row['strategy'], label))

In [ ]:
# [NB03-C14] COMPARING THE DISCOVERY ALGORITHMS (ALPHA, HEURISTICS, INDUCTIVE)
# What: discover a model on the SAME train log with three different miners
#       and compare them on the four quality pillars, on the test set.
# Why: the course (Module 3.1, Case5, PDexample) never trusts a single
#      algorithm: each miner makes different assumptions, and the comparison
#      shows which one suits THIS log. Same input, same evaluation procedure:
#      the only thing that changes is the algorithm.
# Expected: Alpha struggles with the loops of the clinical activities;
#      Heuristics tolerates noise; Inductive guarantees a sound model.

miner_rows = []

# Alpha Miner: the original algorithm; it cannot handle loops and noise well
alpha_net, alpha_im, alpha_fm = pm4py.discover_petri_net_alpha(train_log)
alpha_scores = evaluate_model(test_log, alpha_net, alpha_im, alpha_fm)
alpha_scores['miner'] = 'Alpha'
miner_rows.append(alpha_scores)

# Heuristics Miner: frequency-based, tolerates noise and short loops
heu_net, heu_im, heu_fm = pm4py.discover_petri_net_heuristics(train_log, dependency_threshold=0.9)
heu_scores = evaluate_model(test_log, heu_net, heu_im, heu_fm)
heu_scores['miner'] = 'Heuristics'
miner_rows.append(heu_scores)

# Inductive Miner: guarantees a sound model, used for all the strategies above
ind_net, ind_im, ind_fm = discover_inductive(train_log, 0.1)
ind_scores = evaluate_model(test_log, ind_net, ind_im, ind_fm)
ind_scores['miner'] = 'Inductive'
miner_rows.append(ind_scores)

miner_table = pd.DataFrame(miner_rows)[['miner', 'fitness', 'precision', 'generalization', 'simplicity', 'f1', 'transitions', 'places', 'arcs']]
miner_table.to_csv('miner_comparison.csv', index=False)
print('Saved miner_comparison.csv')

# Heatmap of the four quality pillars per miner (course-style visualisation):
# a compact way to read all four dimensions of every algorithm at once
pillars = miner_table.set_index('miner')[['fitness', 'precision', 'generalization', 'simplicity']]
plt.figure(figsize=(7, 3.2))
sns.heatmap(pillars, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0.5, vmax=1.0, cbar_kws={'label': 'score'})
plt.title('Discovery algorithms on the four quality pillars')
plt.ylabel('')
plt.tight_layout()
plt.savefig('figures/NB03_miner_heatmap.png', dpi=200)
plt.show()

for row in miner_rows:
    print('{:11s} -> fitness {:.3f} | precision {:.3f} | generalization {:.3f} | simplicity {:.3f} | transitions {} | arcs {}'.format(
        row['miner'], row['fitness'], row['precision'], row['generalization'], row['simplicity'],
        row['transitions'], row['arcs']))

# The metrics alone can be fooled, so we also LOOK at the structure: a
# transition with no input place is always enabled and can fire at any time,
# which silently inflates fitness. We count them for each miner.
def count_disconnected(net):
    disconnected = []
    for transition in net.transitions:
        if len(transition.in_arcs) == 0 or len(transition.out_arcs) == 0:
            if transition.label is not None:
                disconnected.append(transition.label)
    return disconnected

print()
for miner_name, net in [('Alpha', alpha_net), ('Heuristics', heu_net), ('Inductive', ind_net)]:
    loose = count_disconnected(net)
    print('{:11s} -> activities left disconnected (always enabled): {}'.format(miner_name, loose if loose else 'none'))

# Save the Alpha net as a figure: the report shows WHY its metrics lie
pm4py.save_vis_petri_net(alpha_net, alpha_im, alpha_fm, 'figures/NB03_alpha_model.png')

miner_table

# Interpretation: Alpha reaches fitness 1.000 AND simplicity 1.000 - the best
# of three miners on two pillars out of four - but it leaves the three
# clinical activities disconnected from the net. A disconnected transition is
# always enabled, so the model replays everything (fitness 1.000) while
# constraining almost nothing (precision 0.594), and it looks "simple" only
# because it has almost no arcs. This is the known Alpha limitation with
# loops, and it is the clearest possible argument for the course rule: never
# judge a model on one metric, use all four pillars AND look at the net.
# Heuristics goes the opposite way (20 transitions, 46 arcs, simplicity
# 0.533): a spaghetti no ED manager could read. Inductive is the balanced
# choice and is the miner used for the strategies of NB03-C13.

In [ ]:
# [NB03-C15] THE EMPIRICAL REFERENCE MODEL
# What: keep the winning model as the empirical reference model and
#       visualise its Petri net.
# Why: no external clinical protocol is provided, so conformance cannot
#      measure clinical correctness. The winning model becomes an EMPIRICAL
#      reference: it describes the typical behaviour of the ED, and
#      conformance below measures how far a stay departs from that typical
#      behaviour - not from a norm.
# Expected: a compact, readable Petri net of the ED process.

# Select the winning model discovered above
winning_models = {'1 - all train stays (noise=0.1)': model1,
                  '2 - top-20 variants (F1)': model2,
                  '3 - head segment (n>=20)': model3}
ref_net, ref_im, ref_fm = winning_models[winner_name]
print('Empirical reference model: {}'.format(winner_name))

# Save and view the Petri net of the reference model
pm4py.save_vis_petri_net(ref_net, ref_im, ref_fm, 'figures/NB03_reference_model.png')
pm4py.view_petri_net(ref_net, ref_im, ref_fm)

# Interpretation: this model is the yardstick for the conformance analysis.
# It expresses typical ED behaviour, not a clinical or regulatory standard -
# a distinction stated explicitly in the report.

In [ ]:
# [NB03-C16] TOKEN-BASED REPLAY ON THE FULL TEST SET
# What: replay every test stay on the reference model and read the per-case
#       fitness, to see how many stays the typical model can reproduce.
# Why: token-based replay is the fast conformance screen (course, Case5):
#      it flags the stays that do not fit before the more expensive
#      alignment analysis.
# Expected: most stays fully fit; a minority with fitness < 1 are the
#      candidates for the deviation analysis.

# Per-case token-replay diagnostics on the test set
tbr = pm4py.conformance_diagnostics_token_based_replay(test_log, ref_net, ref_im, ref_fm,
                                                       return_diagnostics_dataframe=True)

# How many test stays fit the reference model perfectly?
fitting = (tbr['trace_fitness'] >= 0.999).sum()
print('Test stays: {}'.format(len(tbr)))
print('Perfectly fitting stays: {} ({:.1f}%)'.format(fitting, fitting / len(tbr) * 100))
print('Stays with fitness < 1 (non-conforming): {} ({:.1f}%)'.format(
    len(tbr) - fitting, (len(tbr) - fitting) / len(tbr) * 100))
print('\nTrace fitness distribution:')
print(tbr['trace_fitness'].describe().round(3))

# Interpretation: the non-conforming minority is where the process departs
# from its own typical behaviour; the alignments below explain HOW.

In [ ]:
# [NB03-C17] ALIGNMENT DIAGNOSTICS: SYNCHRONOUS, LOG AND MODEL MOVES
# What: compute alignments on the test set and classify every move into the
#       categories of the course (Module 3.2): synchronous moves, log moves
#       and model moves - keeping model moves on INVISIBLE (tau) transitions
#       separate, because they are not deviations.
# Why: the raw pm4py alignment cost mixes two units: a non-synchronous move
#      on a VISIBLE activity costs 10,000, while a model move on an
#      invisible transition costs 1. A raw median cost of 8 therefore means
#      "zero real deviations plus 8 silent routing steps", NOT "8
#      deviations": reporting the raw cost would mislead the reader. We
#      decompose it into readable counts, and we VERIFY the cost identity
#      instead of assuming it.
# Expected: the identity holds on every test case; most stays with zero
#      real deviations.

# Per-case alignment diagnostics as a dataframe (case_id, cost, fitness, is_fit)
align_df = pm4py.conformance_diagnostics_alignments(test_log, ref_net, ref_im, ref_fm,
                                                    return_diagnostics_dataframe=True)

# The alignment MOVES are not in the dataframe: fetch them as a list (Case5)
align_list = pm4py.conformance_diagnostics_alignments(test_log, ref_net, ref_im, ref_fm)

# pm4py returns every alignment step as a tuple (log_step, model_step):
#   ('A', 'A')   -> SYNCHRONOUS move: log and model agree
#   ('A', '>>')  -> LOG MOVE: activity A was performed but the model did not
#                   allow it at that point (an unexpected/extra step)
#   ('>>', 'A')  -> MODEL MOVE on a visible activity: the model expected A
#                   but the log does not have it (a skipped step)
#   ('>>', None) -> model move on an INVISIBLE (tau) transition: internal
#                   routing of the model, NOT a deviation
def classify_moves(alignment):
    counts = {'synchronous': 0, 'log_move': 0, 'model_move': 0, 'tau_move': 0}
    log_move_activities = []
    model_move_activities = []
    for log_step, model_step in alignment['alignment']:
        if log_step != '>>' and model_step != '>>':
            counts['synchronous'] += 1
        elif model_step == '>>':
            counts['log_move'] += 1
            log_move_activities.append(log_step)
        elif model_step is None:
            counts['tau_move'] += 1
        else:
            counts['model_move'] += 1
            model_move_activities.append(model_step)
    return counts, log_move_activities, model_move_activities

# Classify every test case and check the cost identity
move_rows = []
log_move_counter = {}
model_move_counter = {}
identity_holds = 0
for alignment in align_list:
    counts, log_acts, model_acts = classify_moves(alignment)
    deviations = counts['log_move'] + counts['model_move']
    # Verify: cost = 10,000 * visible non-sync moves + 1 * tau moves
    if alignment['cost'] == deviations * 10000 + counts['tau_move']:
        identity_holds += 1
    move_rows.append({'cost': alignment['cost'], 'deviations': deviations,
                      'log_moves': counts['log_move'], 'model_moves': counts['model_move'],
                      'tau_moves': counts['tau_move'], 'synchronous': counts['synchronous']})
    for activity in log_acts:
        log_move_counter[activity] = log_move_counter.get(activity, 0) + 1
    for activity in model_acts:
        model_move_counter[activity] = model_move_counter.get(activity, 0) + 1
moves = pd.DataFrame(move_rows)

print('Cost identity (10,000 x visible moves + 1 x tau moves) verified on {}/{} cases'.format(
    identity_holds, len(moves)))
print()
print('Stays with ZERO real deviations: {} of {} ({:.1f}%)'.format(
    (moves['deviations'] == 0).sum(), len(moves), (moves['deviations'] == 0).mean() * 100))
print('Real deviations per stay: median {:.0f}, p90 {:.0f}, max {:.0f}'.format(
    moves['deviations'].median(), moves['deviations'].quantile(0.90), moves['deviations'].max()))
print()
print('Move totals across the test set:')
print('  synchronous moves : {}'.format(int(moves['synchronous'].sum())))
print('  log moves (activity done but not expected there): {}'.format(int(moves['log_moves'].sum())))
print('  model moves (activity expected but not done)    : {}'.format(int(moves['model_moves'].sum())))
print('  tau moves (silent routing, NOT deviations)      : {}'.format(int(moves['tau_moves'].sum())))
print()
print('Activities involved in LOG moves (done but unexpected):', dict(sorted(log_move_counter.items(), key=lambda x: -x[1])))
print('Activities involved in MODEL moves (expected but skipped):', dict(sorted(model_move_counter.items(), key=lambda x: -x[1])))

# The course (lecture 3.2) further splits deviations into four classes:
# Early/Late (an activity present in BOTH log moves and model moves, ordered
# one way or the other), Insert (log move only) and Skip (model move only).
# With 0 model moves in this log, Early/Late/Skip are structurally
# impossible - they all require a model move to exist - so every one of the
# 405 log moves classifies as INSERT by construction, not by choice.
n_log_total = int(moves['log_moves'].sum())
n_model_total = int(moves['model_moves'].sum())
print('\nFour-class taxonomy (lecture 3.2): Early={}, Late={}, Insert={}, Skip={}'.format(
    0, 0, n_log_total, n_model_total))
print('Early/Late/Skip are 0 by construction (they require a model move; this log has none).')

# Interpretation: the deviation count - not the raw cost - is the readable
# measure. The direction matters: a LOG move means the ED did something the
# typical model does not foresee at that point (extra clinical work), while a
# MODEL move means a step of the typical path was not performed.

In [ ]:
# [NB03-C18] DEVIATIONS BY CONTEXT GROUP
# What: attach the DEVIATION COUNT (not the raw cost) of each test stay to
#       its patient profile and compare across groups.
# Why: a deviation that concentrates on a specific profile is an operational
#      signal (course goal: which deviations, for which profiles). This links
#      the conformance result back to the business segments of NB02.
# Expected: more deviations on the long-tail stays than on the head ones.

# Rebuild a per-case index for the diagnostics (the dataframe carries case_id)
align_df = align_df.reset_index(drop=True)
if 'case_id' in align_df.columns:
    align_df = align_df.set_index('case_id')
    align_df.index = align_df.index.astype(str)
else:
    align_df.index = test_log['case:concept:name'].drop_duplicates().to_numpy()[:len(align_df)]

# Decompose the cost into interpretable columns (verified in NB03-C17)
align_df['deviations'] = align_df['cost'] // 10000      # visible non-synchronous moves
align_df['tau_moves'] = align_df['cost'] % 10000        # silent routing steps, not deviations

# Join onto the case table (test stays only)
test_cases = case_table.loc[[cid for cid in test_ids if cid in align_df.index]].copy()
test_cases['deviations'] = align_df['deviations']
test_cases['alignment_fitness'] = align_df['fitness']

# Segment label for each stay (head / torso / long tail), no data filtered
def stay_segment(case_id):
    if case_id in set(segment_a.index):
        return 'A - head'
    if case_id in set(segment_c.index):
        return 'C - long tail'
    return 'B - torso'
test_cases['variant_segment'] = test_cases.index.map(stay_segment)

# Deviations by variant segment: size, median and share of deviating stays
print('Deviations by variant segment:')
by_seg = test_cases.groupby('variant_segment')['deviations'].agg(['size', 'median', 'mean'])
by_seg['pct_with_deviations'] = test_cases.groupby('variant_segment')['deviations'].apply(lambda x: round((x > 0).mean() * 100, 1))
print(by_seg.round(2))

print('\nDeviations by disposition:')
by_disp = test_cases.groupby('disposition')['deviations'].agg(['size', 'median', 'mean'])
by_disp['pct_with_deviations'] = test_cases.groupby('disposition')['deviations'].apply(lambda x: round((x > 0).mean() * 100, 1))
print(by_disp.sort_values('mean', ascending=False).round(2))

print('\nDeviations, interrupted vs regular pathways:')
print(test_cases.groupby('interrupted_pathway')['deviations'].agg(['size', 'median', 'mean']).round(2))

# Save the per-case conformance table for the report
test_cases[['deviations', 'alignment_fitness', 'variant_segment', 'disposition', 'acuity', 'interrupted_pathway']].to_csv('conformance_by_case_test.csv')
print('\nSaved conformance_by_case_test.csv')

In [ ]:
# [NB03-C19] EXPECTED VS OBSERVED, SEGMENT BY SEGMENT
# What: measure the model expected from the scenario text [NB03-C04] and the
#       discovered reference model [NB03-C15] separately on each variant
#       segment (head, torso, long tail), and read the long-tail deviations
#       against the expected model.
# Why: the business as-is description must be confronted with the data
#      segment by segment: which segments does the text describe well, and
#      what does the long tail show that the description underestimates?
#      This is where an a-priori description either holds or breaks.
# Expected: the permissive text fits every segment, but says nothing about
#      the time and the extra work the long tail carries.

# Evaluate both models on each segment (segments are views, nothing filtered)
segment_ids = {'A - head': set(segment_a.index),
               'C - long tail': set(segment_c.index)}
segment_ids['B - torso'] = set(case_table.index) - segment_ids['A - head'] - segment_ids['C - long tail']

expected_vs_observed = []
for segment_name in ['A - head', 'B - torso', 'C - long tail']:
    ids = segment_ids[segment_name]
    sub_log = abstract_log[abstract_log['case:concept:name'].isin(ids)]
    exp_scores = evaluate_model(sub_log, expected_net, expected_im, expected_fm)
    ref_scores = evaluate_model(sub_log, ref_net, ref_im, ref_fm)
    expected_vs_observed.append({
        'segment': segment_name,
        'stays': len(ids),
        'median_lead_time_min': round(case_table.loc[list(ids), 'lead_time_min'].median(), 0),
        'expected_fitness': round(exp_scores['fitness'], 3),
        'expected_precision': round(exp_scores['precision'], 3),
        'observed_fitness': round(ref_scores['fitness'], 3),
        'observed_precision': round(ref_scores['precision'], 3)})
segment_models = pd.DataFrame(expected_vs_observed)
segment_models.to_csv('expected_vs_observed_by_segment.csv', index=False)
print('Saved expected_vs_observed_by_segment.csv')
print(segment_models.to_string(index=False))

# What does the long tail do that the scenario text does not foresee? Align
# the long-tail TEST stays against the EXPECTED model and count the moves
tail_test_ids = [cid for cid in test_ids if cid in segment_ids['C - long tail']]
tail_test_log = abstract_log[abstract_log['case:concept:name'].isin(tail_test_ids)]
tail_alignments = pm4py.conformance_diagnostics_alignments(tail_test_log, expected_net, expected_im, expected_fm)
tail_log_moves = {}
tail_deviating = 0
for alignment in tail_alignments:
    counts, log_acts, model_acts = classify_moves(alignment)
    if counts['log_move'] + counts['model_move'] > 0:
        tail_deviating += 1
    for activity in log_acts:
        tail_log_moves[activity] = tail_log_moves.get(activity, 0) + 1

print('\nLong-tail TEST stays: {}'.format(len(tail_test_ids)))
print('Long-tail stays deviating from the EXPECTED (scenario) model: {} ({:.1f}%)'.format(
    tail_deviating, tail_deviating / len(tail_test_ids) * 100 if tail_test_ids else 0))
print('Activities the long tail performs where the scenario text does not foresee them:', tail_log_moves)

# Interpretation: read the table with the median lead time column next to the
# fitness columns. The scenario description is permissive enough to replay
# every segment, so on CONTROL FLOW it is not wrong - but it is silent on the
# two things that actually separate the segments: the time (the long tail
# takes far longer than the head) and the extra clinical work its stays
# absorb. The as-is description therefore underestimates the long tail: it
# describes the steps correctly and the burden not at all, which is why the
# improvement backlog cannot be built from the text alone.

In [ ]:
# [NB03-C20] COMPARATIVE PROCESS MINING: MODELS BY SEGMENT
# What: discover a separate model on clinically meaningful sub-logs and
#       compare their structure and fitness/precision - the comparative
#       process mining of lecture 2.3 (as in Case8ContextBased).
# Why: one global model hides differences between patient groups; comparing
#      per-segment models shows whether urgent and non-urgent, or admitted
#      and discharged, patients follow structurally different processes.
#      Sub-logs use ALL their stays (no filter): they are views, not cuts.
# Expected: admitted / urgent stays with richer models than home / non-urgent.

# Helper: discover on a sub-log and evaluate it ON ITSELF, returning a summary
def compare_sublog(name, case_id_set):
    sub_log = abstract_log[abstract_log['case:concept:name'].isin(case_id_set)]
    net, im, fm = discover_inductive(sub_log, 0.2)   # noise 0.2 as in Case8ContextBased
    scores = evaluate_model(sub_log, net, im, fm)
    return {'group': name, 'stays': len(case_id_set),
            'fitness': round(scores['fitness'], 3), 'precision': round(scores['precision'], 3),
            'activities': scores['activities_in_net'], 'places': scores['places'], 'arcs': scores['arcs']}

# Sub-logs by urgency (acuity 1-3 vs 4-5), using the case-level flag from NB01
urgent_ids = set(case_table[case_table['acuity_1_3']].index)
non_urgent_ids = set(case_table[(~case_table['acuity_1_3']) & case_table['acuity'].notna()].index)

# Sub-logs by disposition (admitted/transfer vs discharged home)
admitted_ids = set(case_table[case_table['disposition'].isin(['ADMITTED', 'TRANSFER'])].index)
home_ids = set(case_table[case_table['disposition'] == 'HOME'].index)

# Build the comparison table
comparative_rows = [
    compare_sublog('acuity 1-3 (urgent)', urgent_ids),
    compare_sublog('acuity 4-5 (non-urgent)', non_urgent_ids),
    compare_sublog('admitted / transfer', admitted_ids),
    compare_sublog('discharged home', home_ids),
]
comparative_table = pd.DataFrame(comparative_rows)
comparative_table.to_csv('comparative_models.csv', index=False)
print('Saved comparative_models.csv\n')

comparative_table

In [ ]:
# [NB03-C21] COMPARATIVE PERFORMANCE TEST BETWEEN SEGMENTS
# What: test whether the number of deviations differs
#       significantly between urgent and non-urgent stays.
# Why: comparative process mining is not only about model structure but
#      about whether the difference is significant (course method). The cost
#      is compared with Mann-Whitney U, consistent with the non-parametric
#      choice of NB01.
# Expected: a significant difference would confirm that urgency changes the
#      conformance profile.

from scipy.stats import mannwhitneyu

# Deviation count of urgent vs non-urgent TEST stays (test set only)
urgent_cost = test_cases[test_cases['acuity'].isin([1.0, 2.0, 3.0])]['deviations'].dropna()
non_urgent_cost = test_cases[test_cases['acuity'].isin([4.0, 5.0])]['deviations'].dropna()

print('Urgent stays (acuity 1-3): n={}, median deviations {:.0f}, mean {:.2f}'.format(len(urgent_cost), urgent_cost.median(), urgent_cost.mean()))
print('Non-urgent stays (acuity 4-5): n={}, median deviations {:.0f}, mean {:.2f}'.format(len(non_urgent_cost), non_urgent_cost.median(), non_urgent_cost.mean()))

stat, p_value = mannwhitneyu(urgent_cost, non_urgent_cost, alternative='two-sided')
print('Mann-Whitney U = {:.0f}, p-value = {:.4f}'.format(stat, p_value))
if p_value < 0.05:
    print('The deviation severity differs significantly between urgent and non-urgent stays.')
else:
    print('No significant difference in deviation severity between the two groups.')

In [ ]:
# [NB03-C22] EXPORTS AND NO-DATA-LOSS CHECK
# What: recap the artifacts of this notebook and verify that no stay was
#       lost anywhere in the discovery and conformance analysis.
# Why: project rule - nothing is ever dropped; the train/test split is a
#      hold-out, and every stay is present in exactly one of the two halves.
# Expected: train + test = 1,820 stays, 16,826 events.

# No-data-loss check
print('Stays in train + test: {} (started with {})'.format(len(train_ids) + len(test_ids), len(case_table)))
print('Events in train + test: {} (abstract view has {})'.format(len(train_log) + len(test_log), len(abstract_log)))

# Recap of the artifacts
print('\nNB03 outputs: model_selection_noise.csv, model_selection_topk.csv,')
print('model_selection_candidates.csv, model_selection_test.csv,')
print('miner_comparison.csv (+ heatmap figure), conformance_by_case_test.csv, comparative_models.csv,')
print('expected_vs_observed_by_segment.csv,')
print('reference model figure in figures/NB03_reference_model.png')
print('\nThe empirical reference model was chosen by TEST F1 among three')
print('candidates compared transparently. No stay, variant or event was')
print('removed from the project at any point.')